In [1]:
import torch as t
from torch import Tensor
from jaxtyping import Float

from nnsight import LanguageModel

import sys

sys.path.append("../../../")

from nngine import alter

device = "cuda"

In [2]:
import torch as t
from torch import Tensor
from jaxtyping import Float
from tqdm import tqdm
import numpy as np

from nnsight.models.UnifiedTransformer import UnifiedTransformer

from transformer_lens import HookedTransformer

device = "cuda"

model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device=device,
)
tokenizer = model.tokenizer

model.set_use_hook_mlp_in(True)
model.set_use_split_qkv_input(True)
model.set_use_attn_result(True)

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
# nn_model = LanguageModel(
#     'openai-community/gpt2',
#     device_map="cuda:0",
#     dispatch=True,
#     tokenizer=model.tokenizer
# )

# alter(nn_model)


# clean = tokenizer.encode("When Alan and Alex got a drink at the store, Alex decided to give it to")

# with nn_model.trace(clean):

#     nn_model.transformer.layers[0].attn.split_q.output.save()
#     resid_mid = nn_model.transformer.layers[0].mlp.mlp_in.output.grad.save()

#     logits = nn_model.output.logits.save()
#     value = logits[:,-1,:].sum()
#     value.backward()


In [4]:
from ioi_dataset import IOIDataset

N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

In [5]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()


with t.no_grad():
    # with model.trace(clean_dataset.toks):
    #     clean_logits = model.output.logits.save()

    # with model.trace(corr_dataset.toks):
    #     corrupt_logits = model.output.logits.save()

    with model.trace(clean_dataset.toks):
        clean_logits = model.output.save()

    with model.trace(corr_dataset.toks):
        corrupt_logits = model.output.save()

clean_logits = clean_logits.value
corrupt_logits = corrupt_logits.value

clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Clean direction: 2.805180311203003, Corrupt direction: 1.631453514099121
Clean metric: 1.0, Corrupt metric: 0.0


In [6]:
nn_model = LanguageModel(
    'openai-community/gpt2',
    device_map="cuda:0",
    dispatch=True,
    tokenizer=model.tokenizer
)

alter(nn_model)

In [7]:
import eap

import importlib

importlib.reload(eap)

graph = eap.EAP(nn_model.config, components=["head", "mlp"], device="cuda")

In [8]:
graph.run(
    nn_model,
    clean_dataset.toks,
    corr_dataset.toks,
    batch_size=25,
    metric=ioi_metric,
)

In [9]:
edges = graph.top_edges(n=20, format=True)

head.9.9 -> [-0.027] -> head.11.10.q
head.10.7 -> [0.024] -> head.11.10.q
head.5.5 -> [0.021] -> head.8.6.v
head.9.9 -> [-0.02] -> head.10.7.q
head.5.5 -> [-0.018] -> mlp.5
mlp.0 -> [0.018] -> head.6.9.q
mlp.0 -> [-0.015] -> mlp.4
head.5.5 -> [-0.014] -> head.6.9.q
head.3.0 -> [-0.013] -> mlp.5
mlp.0 -> [-0.012] -> head.11.10.k
mlp.0 -> [-0.012] -> mlp.5
head.9.6 -> [-0.011] -> head.11.10.q
head.5.5 -> [-0.011] -> mlp.6
head.4.11 -> [0.011] -> head.6.9.k
head.3.0 -> [-0.01] -> mlp.4
mlp.5 -> [-0.01] -> mlp.6
head.9.6 -> [-0.01] -> head.10.7.q
mlp.0 -> [-0.01] -> head.10.7.k
head.5.5 -> [0.009] -> head.7.9.v
mlp.0 -> [-0.009] -> head.7.9.k
